In [12]:
import pandas as pd

# csv_file = 'geant-datasets/esmond data perfsonar-ankara.ulakbim.gov.tr to psmp-gn-bw-lis-pt.geant.org 10-24-2023.csv'
csv_file = "geant-datasets/esmond data perfsonar-sonda.rediris.es to psmp-gn-bw-lis-pt.geant.org 10-24-2023.csv"
df = pd.read_csv(csv_file)
df.head()

,Timestamp,Data,Vazao
0,1692961173,2023-08-25 07:59:33,2.027424e+09
1,1692977431,2023-08-25 12:30:31,2.095054e+09
2,1692977900,2023-08-25 12:38:20,2.112883e+09
3,1692992732,2023-08-25 16:45:32,2.109668e+09
4,1692992794,2023-08-25 16:46:34,2.057309e+09


In [13]:
import pandas as pd

def agregar_por_intervalo(df, coluna_tempo="timestamp", coluna_valor="Vazao", horas=4):
    """
    Agrega os valores de uma série temporal em blocos fixos de N horas.
    Mantém NaN nos intervalos sem medições.

    Parâmetros:
    - df: DataFrame com uma coluna de tempo e uma de valor
    - coluna_tempo: nome da coluna com datetime ou string tipo '2023-04-28 13:54:18'
    - coluna_valor: nome da coluna numérica a agregar
    - horas: tamanho da janela (ex: 4 ou 6)
    """
    df = df.copy()
    df[coluna_tempo] = pd.to_datetime(df[coluna_tempo], errors="coerce")
    df = df.set_index(coluna_tempo).sort_index()

    # Reamostragem por intervalo fixo
    df_resampled = df[coluna_valor].resample(f"{horas}h").mean()

    # Garante que períodos vazios permaneçam como NaN
    df_resampled = df_resampled.asfreq(f"{horas}h")

    print(f"Série agregada a cada {horas} horas ({len(df_resampled)} pontos).")
    return df_resampled


# --- Exemplo de uso ---
# df = pd.read_csv("dados.csv")
# serie_4h = agregar_por_intervalo(df, "DataHora", "Vazao", horas=4)
# serie_6h = agregar_por_intervalo(df, "DataHora", "Vazao", horas=6)
serie_4h = agregar_por_intervalo(df, "Data", "Vazao", horas=4)
print(f"Tamanho do dataset: {df.shape[0]}\n Quantidade de pontos faltantes: {serie_4h.isna().sum()}")
print(serie_4h.head(10))

serie_6h = agregar_por_intervalo(df, "Data", "Vazao", horas=6)
print(f"Tamanho do dataset: {df.shape[0]}\n Quantidade de pontos faltantes: {serie_6h.isna().sum()}")
print(serie_6h.head(10))

Série agregada a cada 4 horas (358 pontos).
Tamanho do dataset: 299
 Quantidade de pontos faltantes: 166
Data
2023-08-25 04:00:00    2.027424e+09
2023-08-25 08:00:00             NaN
2023-08-25 12:00:00    2.103969e+09
2023-08-25 16:00:00    2.083325e+09
2023-08-25 20:00:00             NaN
2023-08-26 00:00:00             NaN
2023-08-26 04:00:00    2.364542e+09
2023-08-26 08:00:00    2.282752e+09
2023-08-26 12:00:00    2.106592e+09
2023-08-26 16:00:00    2.142064e+09
Freq: 4h, Name: Vazao, dtype: float64
Série agregada a cada 6 horas (239 pontos).
Tamanho do dataset: 299
 Quantidade de pontos faltantes: 86
Data
2023-08-25 06:00:00    2.027424e+09
2023-08-25 12:00:00    2.093729e+09
2023-08-25 18:00:00    2.082998e+09
2023-08-26 00:00:00             NaN
2023-08-26 06:00:00    2.323647e+09
2023-08-26 12:00:00    2.107374e+09
2023-08-26 18:00:00    2.122306e+09
2023-08-27 00:00:00    2.271222e+09
2023-08-27 06:00:00             NaN
2023-08-27 12:00:00    2.006715e+09
Freq: 6h, Name: Vazao, 

In [24]:
import numpy as np
import pandas as pd

def intervalo_mais_longo_ate2_falhas(vazao: pd.Series):
    """
    Encontra o intervalo mais longo (em número de pontos) contendo no máximo 2 falhas não consecutivas.
    Falhas são valores NaN ou -1.
    
    Retorna:
        (idx_inicio, idx_fim)
    """
    v = vazao.copy()
    v = v.replace(-1, np.nan)  # trata -1 como falha
    is_fail = v.isna().to_numpy()
    n = len(is_fail)

    melhor_tam = 0
    melhor_inicio = 0
    melhor_fim = 0

    # janelamento deslizante
    for i in range(n):
        falhas = 0
        consecutivas = False

        for j in range(i, n):
            if is_fail[j]:
                falhas += 1
                # verifica se houve falha consecutiva
                if j > i and is_fail[j-1]:
                    consecutivas = True
            if falhas > 2 or consecutivas:
                break  # parou, passou do limite

            tamanho = j - i + 1
            if tamanho > melhor_tam:
                melhor_tam = tamanho
                melhor_inicio = i
                melhor_fim = j

    print(f"Maior intervalo: {melhor_tam} pontos (índices {melhor_inicio}–{melhor_fim})")
    return melhor_inicio, melhor_fim

df = pd.read_csv("intervalos vazao  esmond data psmp-gn-bw-poz-pl.geant.org to pspmp-anella.csuc.cat 10-24-2023.csv")
with pd.option_context('display.max_rows', None):
    print(pd.Series(df["Vazao"]))
intervalo_mais_longo_ate2_falhas(pd.Series(df["Vazao"]))

0     -1.000000e+00
1     -1.000000e+00
2     -1.000000e+00
3      8.855245e+08
4      8.351919e+08
5      8.275890e+08
6      8.947026e+08
7      8.894559e+08
8      8.857858e+08
9      8.988955e+08
10     8.813315e+08
11     8.951360e+08
12     9.269424e+08
13     9.203868e+08
14     8.960100e+08
15     8.949614e+08
16     8.991544e+08
17     8.228636e+08
18     9.130483e+08
19     9.109521e+08
20     8.692720e+08
21     8.540624e+08
22     8.923387e+08
23     9.196029e+08
24     8.983659e+08
25     8.302096e+08
26     8.671738e+08
27     9.062313e+08
28     8.897181e+08
29     8.032106e+08
30     8.742510e+08
31     7.366233e+08
32     8.687465e+08
33     7.903651e+08
34     9.016457e+08
35     8.247074e+08
36     8.603581e+08
37     8.477772e+08
38     9.003772e+08
39     8.716289e+08
40     9.057094e+08
41     8.849995e+08
42     8.737283e+08
43     8.912912e+08
44     8.171034e+08
45     8.988942e+08
46     8.619312e+08
47     9.091162e+08
48     8.695318e+08
49     7.649385e+08


(125, 313)

In [ ]:
missing_rates = [0.1, 0.2, 0.3, 0.4]
random_seed = 42
missing_longest = longest_interval.copy()

missing_dfs = {}
for missing_rate in missing_rates:
    missing_idx = missing_longest.sample(
        frac=missing_rate,
        random_state=random_seed
    ).index
    missing_longest.loc[missing_idx, "Vazao"] = -1
    missing_dfs[missing_rate] = missing_longest

missing_dfs[0.1].head(20)